In [30]:
# Import libraries for data processing
import pandas as pd
import numpy as np

# Import library for train-test splitting
from sklearn.model_selection import GroupShuffleSplit

# Import CatBoost Classifier
from catboost import CatBoostClassifier

# Import evaluation metrics
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score

In [3]:
# Load the EDA dataset
df = pd.read_csv("/Users/vaishnavi/Public/ML_assignments/BookScope-main/Data/Goodreads Books EDA.csv")

# Display first five records
df.head()

,bookId,title,author,description,genres,language,num_pages,num_ratings,num_reviews,avg_rating,rated_1,rated_2,rated_3,rated_4,rated_5,genre_list,genre_count,primary_genre,main_author
0,1,Harry Potter and the Half-Blood Prince,J.K. Rowling,The war against Voldemort is not going well; e...,"Art,Biography,Business,Children's,Christian,Cl...",English,652.0,2553909.0,41470.0,4.57,13147.0,29020.0,174312.0,608825.0,1728605.0,"['Fantasy', 'Young Adult', 'Fiction', 'Magic',...",10,Fantasy,J.K. Rowling
1,2,Harry Potter and the Order of the Phoenix,"J.K. Rowling,Mary Grandpré",There is a door at the end of a silent corrido...,"Art,Biography,Business,Children's,Christian,Cl...",English,870.0,2631427.0,44793.0,4.50,16236.0,41738.0,231438.0,665628.0,1676387.0,"['Fantasy', 'Young Adult', 'Fiction', 'Magic',...",10,Fantasy,J.K. Rowling
2,3,Harry Potter and the Sorcerer's Stone,"J.K. Rowling,Mary Grandpré",Harry Potter's life is miserable. His parents ...,"Art,Biography,Business,Children's,Christian,Cl...",English,309.0,7434783.0,117823.0,4.48,126344.0,147184.0,627803.0,1694206.0,4839246.0,"[""Goodreads' Transgender Books page"", 'Fantasy...",11,Goodreads' Transgender Books page,J.K. Rowling
3,4,Harry Potter and the Chamber of Secrets,J.K. Rowling,The Dursleys were so mean and hideous that sum...,"Art,Biography,Business,Children's,Christian,Cl...",English,352.0,2878196.0,55839.0,4.43,15623.0,55060.0,315917.0,787181.0,1704415.0,"['Fantasy', 'Young Adult', 'Fiction', 'Magic',...",10,Fantasy,J.K. Rowling
4,5,Harry Potter and the Prisoner of Azkaban,"J.K. Rowling,Mary Grandpré","For twelve long years, the dread fortress of A...","Art,Biography,Business,Children's,Christian,Cl...",English,435.0,2972519.0,58483.0,4.57,14124.0,29105.0,214151.0,701765.0,2013374.0,"['Fantasy', 'Young Adult', 'Fiction', 'Magic',...",10,Fantasy,J.K. Rowling


In [4]:
# Display dataset shape
print("Dataset Shape:", df.shape)

# Display column names
print("\nColumns:")
print(df.columns.tolist())

# Display dataset information
df.info()

Dataset Shape: (19941, 19)

Columns:
['bookId', 'title', 'author', 'description', 'genres', 'language', 'num_pages', 'num_ratings', 'num_reviews', 'avg_rating', 'rated_1', 'rated_2', 'rated_3', 'rated_4', 'rated_5', 'genre_list', 'genre_count', 'primary_genre', 'main_author']
<class 'pandas.DataFrame'>
RangeIndex: 19941 entries, 0 to 19940
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   bookId         19941 non-null  int64  
 1   title          19941 non-null  str    
 2   author         19941 non-null  str    
 3   description    19941 non-null  str    
 4   genres         19941 non-null  str    
 5   language       19941 non-null  str    
 6   num_pages      19941 non-null  float64
 7   num_ratings    19941 non-null  float64
 8   num_reviews    19941 non-null  float64
 9   avg_rating     19941 non-null  float64
 10  rated_1        19941 non-null  float64
 11  rated_2        19941 non-null  float64
 12  rate

In [5]:
# Check missing values in each column
print(df.isnull().sum())

# Check duplicate book IDs
print("Duplicate Book IDs:", df['bookId'].duplicated().sum())

bookId           0
title            0
author           0
description      0
genres           0
language         0
num_pages        0
num_ratings      0
num_reviews      0
avg_rating       0
rated_1          0
rated_2          0
rated_3          0
rated_4          0
rated_5          0
genre_list       0
genre_count      0
primary_genre    0
main_author      0
dtype: int64
Duplicate Book IDs: 0


In [6]:
# Check for invalid numerical values
print("Negative rating counts:", (df['num_ratings'] < 0).sum())
print("Invalid page counts:", (df['num_pages'] <= 0).sum())

# Display descriptive statistics
df[['num_pages', 'num_ratings']].describe()

Negative rating counts: 0
Invalid page counts: 643


,num_pages,num_ratings
count,19941.000000,1.994100e+04
mean,289.252495,7.238187e+04
std,295.282735,3.268402e+05
min,0.000000,0.000000e+00
25%,160.000000,2.200000e+01
50%,252.000000,4.370000e+02
75%,356.000000,1.039400e+04
max,23931.000000,7.444102e+06


In [7]:
# Remove records with negative rating counts
df = df[df['num_ratings'] >= 0].copy()

# Replace invalid page counts with NaN
df.loc[df['num_pages'] <= 0, 'num_pages'] = np.nan

# Reset index
df = df.reset_index(drop=True)

print("Dataset Shape:", df.shape)

Dataset Shape: (19941, 19)


In [8]:
# Calculate the number of words in the book title
df['title_length'] = df['title'].astype(str).str.split().str.len()

# Calculate the number of words in the book description
df['description_length'] = df['description'].astype(str).str.split().str.len()

# Display newly created features
df[['title', 'title_length', 'description_length']].head()

,title,title_length,description_length
0,Harry Potter and the Half-Blood Prince,6,205
1,Harry Potter and the Order of the Phoenix,8,204
2,Harry Potter and the Sorcerer's Stone,6,285
3,Harry Potter and the Chamber of Secrets,7,222
4,Harry Potter and the Prisoner of Azkaban,7,176


In [15]:
# Define categorical features
categorical_features = [
    'main_author',
    'primary_genre',
    'language'
]

# Convert categorical features into string format
for column in categorical_features:
    df[column] = df[column].astype(str).str.strip()

# Display categorical features
df[categorical_features].head()

,main_author,primary_genre,language
0,J.K. Rowling,Fantasy,English
1,J.K. Rowling,Fantasy,English
2,J.K. Rowling,Goodreads' Transgender Books page,English
3,J.K. Rowling,Fantasy,English
4,J.K. Rowling,Fantasy,English


In [10]:
# Display most common genres
print("Primary Genres:")
print(df['primary_genre'].value_counts().head(20))

# Display most common languages
print("\nLanguages:")
print(df['language'].value_counts().head(10))

# Display number of unique authors
print("\nUnique Authors:", df['main_author'].nunique())

Primary Genres:
primary_genre
Unknown            5055
Fiction            2656
Classics           1501
Nonfiction         1474
Fantasy             915
History             604
Childrens           510
Mystery             495
Philosophy          395
Biography           384
Historical          379
Sequential Art      357
Poetry              355
Science Fiction     354
Horror              294
Science             286
Short Stories       249
Plays               246
Business            179
Art                 174
Name: count, dtype: int64

Languages:
language
English               16209
Unknown                2768
Spanish                 447
German                  195
French                  169
Japanese                 47
Multiple languages       24
Italian                  14
Portuguese               13
Chinese                  13
Name: count, dtype: int64

Unique Authors: 8481


In [11]:
# Display rating count statistics
print(df['num_ratings'].describe())

# Calculate 75th percentile
print("75th Percentile:", df['num_ratings'].quantile(0.75))

count    1.994100e+04
mean     7.238187e+04
std      3.268402e+05
min      0.000000e+00
25%      2.200000e+01
50%      4.370000e+02
75%      1.039400e+04
max      7.444102e+06
Name: num_ratings, dtype: float64
75th Percentile: 10394.0


In [12]:
# Define popularity threshold
popularity_threshold = 10000

# Create target variable
df['popularity'] = (
    df['num_ratings'] >= popularity_threshold
).astype(int)

# Display sample records
df[['title', 'num_ratings', 'popularity']].head(10)

,title,num_ratings,popularity
0,Harry Potter and the Half-Blood Prince,2553909.0,1
1,Harry Potter and the Order of the Phoenix,2631427.0,1
2,Harry Potter and the Sorcerer's Stone,7434783.0,1
3,Harry Potter and the Chamber of Secrets,2878196.0,1
4,Harry Potter and the Prisoner of Azkaban,2972519.0,1
5,Harry Potter and the Goblet of Fire,2725467.0,1
6,The Harry Potter Collection,31201.0,1
7,"Harry Potter Boxed Set, Books 1-5 (Harry Potte...",58833.0,1
8,"Unauthorized Harry Potter Book Seven News: \""H...",30.0,0
9,Harry Potter Collection,31200.0,1


In [13]:
# Count books in each popularity class
print(df['popularity'].value_counts())

# Calculate percentage of each class
print("\nPercentage Distribution:")
print(df['popularity'].value_counts(normalize=True) * 100)

popularity
0    14907
1     5034
Name: count, dtype: int64

Percentage Distribution:
popularity
0    74.755529
1    25.244471
Name: proportion, dtype: float64


In [16]:
# Check incorrect Goodreads genre labels
print(df[df['primary_genre'].str.contains(
    'Goodreads', case=False, na=False
)]['primary_genre'].value_counts())

primary_genre
Goodreads' Transgender Books page    7
Goodreads country page               2
Name: count, dtype: int64


In [17]:
# Replace Goodreads page labels with Unknown
df.loc[
    df['primary_genre'].str.contains(
        'Goodreads', case=False, na=False
    ),
    'primary_genre'
] = 'Unknown'

# Verify updated genre values
print(df['primary_genre'].value_counts().head(20))

primary_genre
Unknown            5064
Fiction            2656
Classics           1501
Nonfiction         1474
Fantasy             915
History             604
Childrens           510
Mystery             495
Philosophy          395
Biography           384
Historical          379
Sequential Art      357
Poetry              355
Science Fiction     354
Horror              294
Science             286
Short Stories       249
Plays               246
Business            179
Art                 174
Name: count, dtype: int64


In [19]:
# Select independent variables
features = [
    'main_author',
    'primary_genre',
    'language',
    'num_pages',
    'title_length',
    'description_length'
]

X = df[features].copy()

# Select dependent variable
y = df['popularity']

# Display updated input features
X.head()

,main_author,primary_genre,language,num_pages,title_length,description_length
0,J.K. Rowling,Fantasy,English,652.0,6,205
1,J.K. Rowling,Fantasy,English,870.0,8,204
2,J.K. Rowling,Unknown,English,309.0,6,285
3,J.K. Rowling,Fantasy,English,352.0,7,222
4,J.K. Rowling,Fantasy,English,435.0,7,176


In [20]:
from sklearn.model_selection import GroupShuffleSplit

# Group books based on title and author
groups = (
    df['title'].astype(str).str.lower().str.strip()
    + '|'
    + df['main_author'].str.lower().str.strip()
)

# Initialize train-test splitter
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

# Generate training and testing indices
train_idx, test_idx = next(
    splitter.split(X, y, groups)
)

# Create training dataset
X_train = X.iloc[train_idx]
y_train = y.iloc[train_idx]

# Create testing dataset
X_test = X.iloc[test_idx]
y_test = y.iloc[test_idx]

# Display dataset dimensions
print("X_train Shape:", X_train.shape)
print("X_test Shape:", X_test.shape)
print("y_train Shape:", y_train.shape)
print("y_test Shape:", y_test.shape)

X_train Shape: (16083, 6)
X_test Shape: (3858, 6)
y_train Shape: (16083,)
y_test Shape: (3858,)


In [21]:
# Check training class distribution
print("Training Class Distribution:")
print(y_train.value_counts())

# Check testing class distribution
print("\nTesting Class Distribution:")
print(y_test.value_counts())

# Check for overlapping book groups
overlap = set(groups.iloc[train_idx]).intersection(
    set(groups.iloc[test_idx])
)

print("\nOverlapping Book Groups:", len(overlap))

Training Class Distribution:
popularity
0    12025
1     4058
Name: count, dtype: int64

Testing Class Distribution:
popularity
0    2882
1     976
Name: count, dtype: int64

Overlapping Book Groups: 0


In [22]:
# Check percentage distribution in training data
print("Training Class Percentage:")
print(y_train.value_counts(normalize=True) * 100)

# Check percentage distribution in testing data
print("\nTesting Class Percentage:")
print(y_test.value_counts(normalize=True) * 100)

Training Class Percentage:
popularity
0    74.768389
1    25.231611
Name: proportion, dtype: float64

Testing Class Percentage:
popularity
0    74.701918
1    25.298082
Name: proportion, dtype: float64


Initializing Catboost Model

In [23]:
# Initialize CatBoost classification model
model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=50,
    allow_writing_files=False
)

In [24]:
# Define categorical features
categorical_features = [
    'main_author',
    'primary_genre',
    'language'
]

# Train the CatBoost model
model.fit(
    X_train,
    y_train,
    cat_features=categorical_features
)

print("Model training completed successfully!")

0:	learn: 0.6545092	total: 65.4ms	remaining: 19.6s
50:	learn: 0.3316823	total: 328ms	remaining: 1.6s
100:	learn: 0.3180133	total: 625ms	remaining: 1.23s
150:	learn: 0.3106355	total: 919ms	remaining: 907ms
200:	learn: 0.3046482	total: 1.21s	remaining: 597ms
250:	learn: 0.2980964	total: 1.51s	remaining: 295ms
299:	learn: 0.2915469	total: 1.8s	remaining: 0us
Model training completed successfully!


In [25]:
# Check whether the model is trained
print("Model Trained:", model.is_fitted())

# Display number of trained trees
print("Number of Trees:", model.tree_count_)

# Display model classes
print("Model Classes:", model.classes_)

# Display training loss
print("Training Loss:", model.get_best_score())

Model Trained: True
Number of Trees: 300
Model Classes: [0 1]
Training Loss: {'learn': {'Logloss': 0.2915468546834976}}


Model Testing

In [26]:
# Predict popularity using testing data
y_pred = model.predict(X_test).ravel()

# Display first 10 predicted values
print("Predicted Values:")
print(y_pred[:10])

# Display first 10 actual values
print("\nActual Values:")
print(y_test.iloc[:10].values)

Predicted Values:
[1 1 1 1 1 0 0 0 0 0]

Actual Values:
[1 0 0 0 1 1 0 0 0 0]


In [27]:
# Predict probability scores for the popular class
y_prob = model.predict_proba(X_test)[:, 1]

# Display first 10 probability scores
print("Popularity Scores:")
print(y_prob[:10])

Popularity Scores:
[0.88770858 0.88552988 0.88930969 0.93885626 0.91941298 0.44945577
 0.13697354 0.10542954 0.06271235 0.3387388 ]


In [28]:
# Create a dataframe to compare predictions
prediction_results = pd.DataFrame({
    'Book Title': df.iloc[test_idx]['title'].values,
    'Actual Popularity': y_test.values,
    'Predicted Popularity': y_pred,
    'Popularity Score': y_prob
})

# Display first 10 prediction results
prediction_results.head(10)

,Book Title,Actual Popularity,Predicted Popularity,Popularity Score
0,Harry Potter and the Sorcerer's Stone,1,1,0.887709
1,The Hitchhiker's Guide to the Galaxy: Quandary...,0,1,0.885530
2,The Hitchhiker's Guide to the Galaxy: Quintess...,0,1,0.889310
3,Bill Bryson's African Diary,0,1,0.938856
4,Neither Here nor There: Travels in Europe,1,1,0.919413
5,J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...,1,0,0.449456
6,The Lord of the Rings,0,0,0.136974
7,Agile Web Development with Rails: A Pragmatic ...,0,0,0.105430
8,A Changeling for All Seasons,0,0,0.062712
9,Changeling,0,0,0.338739


In [29]:
# Count correct predictions
correct_predictions = (y_test.values == y_pred).sum()

# Count incorrect predictions
incorrect_predictions = (y_test.values != y_pred).sum()

# Display prediction results
print("Total Test Records:", len(y_test))
print("Correct Predictions:", correct_predictions)
print("Incorrect Predictions:", incorrect_predictions)

Total Test Records: 3858
Correct Predictions: 3330
Incorrect Predictions: 528


In [31]:
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

# Calculate precision
precision = precision_score(y_test, y_pred)

# Calculate recall
recall = recall_score(y_test, y_pred)

# Calculate F1 score
f1 = f1_score(y_test, y_pred)

# Calculate ROC AUC
roc_auc = roc_auc_score(y_test, y_prob)

# Calculate average precision
avg_precision = average_precision_score(y_test, y_prob)

# Display evaluation metrics
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall: {recall * 100:.2f}%")
print(f"F1 Score: {f1 * 100:.2f}%")
print(f"ROC AUC: {roc_auc:.4f}")
print(f"Average Precision: {avg_precision:.4f}")

Accuracy: 86.31%
Precision: 69.05%
Recall: 83.20%
F1 Score: 75.46%
ROC AUC: 0.9264
Average Precision: 0.8268


In [32]:
# Display detailed classification report
print(classification_report(
    y_test,
    y_pred,
    target_names=['Less Popular', 'Popular']
))

              precision    recall  f1-score   support

Less Popular       0.94      0.87      0.91      2882
     Popular       0.69      0.83      0.75       976

    accuracy                           0.86      3858
   macro avg       0.81      0.85      0.83      3858
weighted avg       0.88      0.86      0.87      3858



In [33]:
# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Display confusion matrix
print("Confusion Matrix:")
print(cm)

# Extract confusion matrix values
tn, fp, fn, tp = cm.ravel()

print("\nTrue Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)

Confusion Matrix:
[[2518  364]
 [ 164  812]]

True Negatives: 2518
False Positives: 364
False Negatives: 164
True Positives: 812


In [34]:
# Predict popularity for training data
y_train_pred = model.predict(X_train).ravel()

# Calculate training accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)

# Compare training and testing accuracy
print(f"Training Accuracy: {train_accuracy * 100:.2f}%")
print(f"Testing Accuracy: {accuracy * 100:.2f}%")

# Calculate difference
print(
    f"Accuracy Difference: "
    f"{(train_accuracy - accuracy) * 100:.2f} percentage points"
)

Training Accuracy: 92.82%
Testing Accuracy: 86.31%
Accuracy Difference: 6.51 percentage points


In [35]:
# Predict every book as less popular
baseline_predictions = np.zeros(len(y_test), dtype=int)

# Calculate baseline accuracy
baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

# Compare baseline and CatBoost accuracy
print(f"Baseline Accuracy: {baseline_accuracy * 100:.2f}%")
print(f"CatBoost Accuracy: {accuracy * 100:.2f}%")

print(
    f"Accuracy Improvement: "
    f"{(accuracy - baseline_accuracy) * 100:.2f} percentage points"
)

Baseline Accuracy: 74.70%
CatBoost Accuracy: 86.31%
Accuracy Improvement: 11.61 percentage points


Calculate feature importance

In [36]:
# Get feature importance from the trained model
importance = model.get_feature_importance()

# Create a dataframe for feature importance
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': importance
})

# Sort features based on importance
feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
).reset_index(drop=True)

# Display feature importance
feature_importance

,Feature,Importance
0,primary_genre,49.734775
1,main_author,14.198332
2,language,12.327438
3,num_pages,10.644601
4,title_length,7.534012
5,description_length,5.560842


Create a prediction function

In [37]:
# Function to predict the popularity of an upcoming book
def predict_book_popularity(
    title,
    author,
    genre,
    language,
    num_pages,
    description
):

    # Calculate title and description length
    title_length = len(title.split())
    description_length = len(description.split())

    # Clean genre value using the same rule as training
    genre = genre.strip()

    if 'goodreads' in genre.lower():
        genre = 'Unknown'

    # Handle invalid page counts
    if num_pages is None or num_pages <= 0:
        num_pages = np.nan

    # Create input dataframe
    new_book = pd.DataFrame({
        'main_author': [author.strip()],
        'primary_genre': [genre],
        'language': [language.strip()],
        'num_pages': [num_pages],
        'title_length': [title_length],
        'description_length': [description_length]
    })

    # Maintain the same feature order as training
    new_book = new_book[X_train.columns]

    # Predict popularity class
    prediction = model.predict(new_book).ravel()[0]

    # Predict popularity score
    probability = model.predict_proba(new_book)[0][1]

    # Display book details and prediction
    print("Book Title:", title)
    print("Author:", author)
    print("Genre:", genre)
    print("Language:", language)
    print("Number of Pages:", num_pages)

    if prediction == 1:
        print("\nPrediction: Popular")
    else:
        print("\nPrediction: Less Popular")

    print(f"Popularity Score: {probability * 100:.2f}%")

In [38]:
# Predict popularity of an example upcoming book
predict_book_popularity(
    title="The Secrets of the Hidden Kingdom",
    author="J.K. Rowling",
    genre="Fantasy",
    language="English",
    num_pages=450,
    description="""
    A young explorer discovers a hidden magical kingdom
    and embarks on an adventurous journey filled with
    mystery, friendship and unexpected challenges.
    """
)

Book Title: The Secrets of the Hidden Kingdom
Author: J.K. Rowling
Genre: Fantasy
Language: English
Number of Pages: 450

Prediction: Popular
Popularity Score: 92.96%


In [39]:
# Predict popularity of a book by a new author
predict_book_popularity(
    title="The Journey Within",
    author="Emily Parker",
    genre="Fiction",
    language="English",
    num_pages=280,
    description="""
    A young woman embarks on a journey of self-discovery
    while facing personal challenges and discovering
    the importance of friendship and courage.
    """
)

Book Title: The Journey Within
Author: Emily Parker
Genre: Fiction
Language: English
Number of Pages: 280

Prediction: Less Popular
Popularity Score: 47.94%
